# Monthly & Annual LST Temporal Analysis

## What This Notebook Does

This notebook extracts and analyzes **Land Surface Temperature (LST)** data from **MODIS Terra (MOD11A1)** and **Aqua (MYD11A1)** satellites using Google Earth Engine. It processes all cities with delineated AOIs from Notebook 01 and performs comprehensive temporal trend analysis to quantify warming/cooling patterns over 2006–2025. Stakeholder perceptions from Notebook 02 are validated against measured trends.

---

## Table of Contents

Navigate directly to any section:

1. [Setup & Authentication](#setup) — Initialize GEE and Dask
2. [Load Configuration](#load-config) — Import settings from `config.py`
3. [Load AOI Data](#load-aoi) — Auto-load city boundaries
4. [LST Extraction with QA Filtering](#lst-extraction) — MODIS LST download with hierarchical QA
5. [Visualize LST Maps](#visualize-maps) — Monthly/annual GeoTIFF maps
6. [Compute LST Statistics](#compute-stats) — Temporal metrics from GeoTIFFs
7. [Trend Analysis & Visualization](#trend-analysis) — Statistical tests and dual-panel plots
8. [Cross-City Heatmap Comparison](#heatmap) — Spatiotemporal heatmaps
9. [Trend Category & Violin Charts](#trend-charts) — Summary charts (9a) and violin distributions (9b)
10. [Stakeholder Validation — Yearly](#validation-yearly) — Compare perceptions vs. measured annual trends
11. [Stakeholder Validation — Monthly](#validation-monthly) — Compare perceptions vs. peak month timing
12. [Summary & Next Steps](#summary)

---

## Sub-Research Objectives

1. **Extract multi-decadal LST time series** (2006–2025) with rigorous quality assurance filtering
2. **Quantify warming/cooling trends** using linear regression and Mann-Kendall non-parametric tests
3. **Characterize seasonal patterns** through monthly climatology analysis
4. **Compare across cities** using cross-city heatmaps sorted by thermal intensity
5. **Validate satellite-derived trends** against stakeholder perceptions of warming

---

## Data Sources

| Dataset | Source | Temporal Coverage | Resolution |
|---------|--------|-------------------|------------|
| **MODIS Terra** | MOD11A1 v6.1 (GEE) | 2006–2025 | 1 km, daily → monthly/yearly |
| **MODIS Aqua** | MYD11A1 v6.1 (GEE) | 2006–2025 | 1 km, daily → monthly/yearly |
| **Stakeholder Survey** | Master Composite (CSV) | 2024 | Per-city workshop responses |

---

## Notebook Structure

### Processing Pipeline

```
┌─────────────────────────────────────────────────────────────┐
│ SECTION 1-2: Setup & Configuration                         │
│ • Initialize GEE and Dask for parallel processing           │
│ • Load project settings from config.py                      │
│ • Set LST time mode (day / night / both)                    │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│ SECTION 3: Load AOI Data                                   │
│ • Auto-discover cities from Notebook 01 outputs             │
│ • Load GeoJSON boundaries and bounding boxes                │
│ • Validate geometry and CRS compatibility                   │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│ SECTION 4: LST Extraction                                  │
│ • Download MODIS LST with hierarchical QA filtering         │
│ • Strict (≤1K error) → Moderate (≤2K) fallback             │
│ • Export monthly climatology + yearly GeoTIFFs              │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│ SECTION 5: Visualization                                   │
│ • Monthly climatology maps (12 panels)                      │
│ • Yearly LST maps (2006–2025, 20 panels)                   │
│ • QA composition maps (Strict/Moderate distribution)        │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│ SECTION 6-7: Statistics & Trend Analysis                   │
│ • Compute mean, std, min, max, median per city/period       │
│ • Linear regression with warming rate (°C/decade)           │
│ • Mann-Kendall trend test with significance                 │
│ • Dual-panel diagnostic plots (annual trend + climatology)  │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│ SECTION 8-9: Cross-City Comparison                         │
│ • Year × City and Month × City heatmaps                    │
│ • Trend category summary chart                              │
│ • Annual LST violin (butterfly) charts                     │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│ SECTION 10-11: Stakeholder Validation                      │
│ • Compare perceived warming vs. measured annual trends      │
│ • Compare perceived peak month vs. LST hottest month       │
│ • Sunburst charts summarizing agreement levels              │
└─────────────────────────────────────────────────────────────┘
```

### Code Organization

- **`gee_utils.py`**: GEE initialization, MODIS LST extraction, AOI loading
- **`temporal_analysis.py`**: Statistics computation, trend analysis, heatmap data preparation
- **`temporal_analysis_utils.py`**: Helper functions for temporal aggregation
- **`visualization.py`**: LST maps, heatmaps, trend plots, validation charts
- **`data_io.py`**: CSV/JSON/GeoTIFF exports

## Output Structure

All results are saved to: `data/outputs/03_trend_analysis_monthly_annual/`

```
03_trend_analysis_monthly_annual/
├── Stats/
│   ├── All_Cities_LST_Statistics.csv
│   ├── City_LST_Enhanced_Summary.csv
│   ├── LST_Yearly_Day_Heatmap_AllCities*.png
│   ├── LST_Monthly_Day_Heatmap_AllCities*.png
│   ├── Trend_Category_Summary_Chart.png/pdf
│   ├── Annual_LST_Violin_Chart.png/pdf
│   ├── Stakeholder_Validation_Sunburst_Chart.png
│   └── Stakeholder_Validation_Monthly_Sunburst_Chart.png
└── City_[Name]/
    └── LST_Temporal_Analysis/
        ├── Day/ (and/or Night/)
        │   ├── GeoTIFF/        # Monthly + yearly LST rasters
        │   ├── Figures/        # Time series and trend plots
        │   └── Statistics/     # Per-city CSV statistics
        └── Cross_City_Statistics/
```

---

## Configuration

Analysis parameters are centralized in **`config.py`**:

```python
LST_CONFIG = {
    'lst_time': 'both',                    # 'day', 'night', or 'both'
    'primary_collection': 'MODIS/061/MYD11A1',
    'qa_levels_to_use': ['strict', 'moderate'],
}
TEMPORAL_CONFIG = {
    'start_date': '2006-01-01',
    'end_date': '2025-12-31',
}
```

Modify `config.py` to adjust LST time, date ranges, and QA thresholds without changing notebook code.

---

## Prerequisites
- ✅ Completed **Notebook 01** (AOI Delineation)
- ✅ Configured `config.py` with valid GEE credentials
- ✅ Activated virtual environment with required packages

---

## Mathematical Background: Trend Analysis

### 1. Linear Regression Model
For each city and time period (day/night), we fit an ordinary least squares (OLS) regression:

$$\text{LST}_t = \beta_0 + \beta_1 \cdot t + \epsilon_t$$

where:
- $\text{LST}_t$ = median LST at time $t$ (°C)
- $t$ = year (2006, 2007, ..., 2025)
- $\beta_0$ = intercept (baseline temperature)
- $\beta_1$ = slope (°C/year)
- $\epsilon_t$ = residual error

### 2. Warming/Cooling Rate Metrics

**Per Decade Rate:**
$$\Delta T_{decade} = 10 \cdot \beta_1$$

**Total Study Period Change:**
$$\Delta T_{total} = \beta_1 \cdot (t_{end} - t_{start})$$

For 2006-2025: $\Delta T_{total} = \beta_1 \cdot 19$

### 3. Statistical Significance

**Coefficient of Determination ($R^2$):**
$$R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{n}(y_i - \bar{y})^2}$$

**P-value Interpretation:**
- $p < 0.05$ → Statistically significant trend (*)
- $p \geq 0.05$ → No significant trend

### 4. Mann-Kendall Trend Test
Non-parametric test for monotonic trends:

$$S = \sum_{i=1}^{n-1} \sum_{j=i+1}^{n} \text{sgn}(x_j - x_i)$$

Test statistic $Z$ follows standard normal distribution under null hypothesis (no trend).

<a id="setup"></a>
## Block 1: Setup and Authentication

**Purpose**: Initialize Google Earth Engine and Dask for parallel processing.

**What This Section Does**:
- Imports core libraries and project modules from `src/`
- Authenticates Google Earth Engine via service account
- Configures Dask distributed scheduler for parallel LST extraction

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# =============================================================================
# IMPORT REQUIRED LIBRARIES
# =============================================================================

# Core libraries
import os
import sys
import json
import time
import re
import glob
import calendar
import warnings
import traceback
import requests
import shutil
from datetime import datetime, timedelta

# Data processing
import numpy as np
import pandas as pd

# Geospatial libraries
import ee                             # Google Earth Engine API
import geemap                         # Interactive mapping with Earth Engine
import geopandas as gpd               # Geospatial data handling
import rasterio                       # Raster data I/O
from rasterio.plot import show        # Plotting raster data
import pyproj                         # Projections and transformations
# Ensure Point is imported for LST extraction
from shapely.geometry import Point

# Statistical analysis
from scipy import stats               # Statistical functions
from scipy.optimize import curve_fit  # Curve fitting
# Use np.inf for infinity, do not import inf from scipy

# Parallel processing with Dask
import dask
from dask.distributed import Client, progress, wait
from dask import delayed
import multiprocessing
from tqdm.notebook import tqdm        # Progress bars

# Visualization
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.cm as cm
from matplotlib.cm import ScalarMappable
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
import seaborn as sns

# Fix for libpysal import error with scipy.inf
# Monkey patch scipy.inf before libpysal is imported
import scipy
if not hasattr(scipy, 'inf'):
    import numpy as np
    scipy.inf = np.inf
    
# Spatial statistics (for hotspot analysis)
import libpysal
from libpysal.weights import Queen, KNN
from esda.getisord import G_Local
import pymannkendall

# Import project configuration and custom modules
sys.path.insert(0, os.path.abspath('../'))
import config
from src import gee_utils, temporal_analysis, visualization

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')

print("✅ Libraries imported successfully")
print(f"📁 Project root: {config.PROJECT_ROOT}")
print(f"🗂️ Data directory: {config.DATA_DIR}")

In [ ]:
# =============================================================================
# INITIALIZE GOOGLE EARTH ENGINE
# =============================================================================

print("="*70)
print("INITIALIZING GOOGLE EARTH ENGINE")
print("="*70)

# Initialize GEE using modularized function
gee_available = gee_utils.initialize_gee(
    config.GEE_CONFIG['service_json'],
    config.GEE_CONFIG['project_id']
)

if gee_available:
    print("✅ Google Earth Engine initialized successfully")
    print(f"🔑 Project ID: {config.GEE_CONFIG['project_id']}")
else:
    print("❌ Google Earth Engine initialization failed")
    print("⚠️ Some features will be limited")

# Store GEE availability for later checks
GEE_AVAILABLE = gee_available

print("="*70)

In [ ]:
# =============================================================================
# INITIALIZE DASK CLIENT (auto-scaled to city count)
# =============================================================================
# Worker count is looked up from config.DASK_WORKERS based on how many cities
# are being processed this run.  Edit config.py → DASK_WORKERS to tune it.
#
# Tier table (cities → workers):
#   ≤  5 → 3   workers   (original safe setting)
#   ≤ 10 → 4   workers
#   ≤ 15 → 5   workers
#   ≤ 20 → 6   workers
#   ≤ 30 → 6   workers
#   ≤ 50 → 7   workers   ← current maximum
#
# If the Dask dashboard shows high memory pressure, lower the count in
# config.DASK_WORKERS and re-run this cell only.

n_cities_this_run = len(config.get_cities_to_process())
n_workers = config.get_dask_workers(n_cities_this_run)
n_threads = 2  # Threads per worker (keep at 2 for GEE I/O-bound tasks)

print("=" * 70)
print("DASK CLIENT SETUP")
print("=" * 70)
print(f"Cities this run  : {n_cities_this_run}")
print(f"Workers selected : {n_workers}  "
      f"(from config.DASK_WORKERS tier ≤ {next(t for t in sorted(config.DASK_WORKERS) if n_cities_this_run <= t)})")
print(f"Threads/worker   : {n_threads}")
print(f"Total threads    : {n_workers * n_threads}")

# Shut down any existing client before creating a new one
try:
    client.close()
    print("ℹ️  Previous Dask client closed.")
except NameError:
    pass

client = Client(n_workers=n_workers, threads_per_worker=n_threads)
print(f"\n✅ Dask client ready")
print(f"   Dashboard: {client.dashboard_link}")


# Helper to print live cluster info
def print_dask_info():
    info = client.scheduler_info()
    print(f"Dask Cluster Information:")
    print(f"  Workers: {len(info['workers'])}")
    print(f"  Total Threads: {sum(w['nthreads'] for w in info['workers'].values())}")
    print(f"  Total Memory: {sum(w['memory_limit'] for w in info['workers'].values()) / 1e9:.2f} GB")


print_dask_info()
print("=" * 70)

<a id="load-config"></a>
## Block 2: Load Configuration

**Purpose**: Load all settings from `config.py` for backward compatibility.

All settings are centralized in [`config.py`](../config.py). This cell creates local references for backward compatibility.

**Key Configuration Parameters:**
- `LST_CONFIG['lst_time']` → 'day', 'night', or 'both'
- `TEMPORAL_CONFIG` → Date ranges (2006–2025), aggregation methods
- `PROCESSING_OPTIONS` → QA tiers (Strict/Moderate), Dask workers
- `VISUALIZATION_CONFIG` → Color schemes, temperature ranges

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# =============================================================================
# LOAD PROJECT CONFIGURATION
# =============================================================================

print("="*70)
print("LOADING PROJECT CONFIGURATION")
print("="*70)

# All configuration is now centralized in config.py
# Users should modify config.py to change behavior

# Create local references for backward compatibility with existing code
# These are REFERENCES, not copies - they point to config.py
PROJECT_CONFIG = {
    'study_year': str(config.PROJECT_SETTINGS['study_year']),
    'start_date': config.TEMPORAL_CONFIG['start_date'],
    'end_date': config.TEMPORAL_CONFIG['end_date'],
    'data_dir': str(config.DATA_DIR),
    'output_dir': str(config.OUTPUT_DATA_PATH),
    'aoi_results_dir': str(config.get_output_dir('01_aoi_delineation')),
    'lst_output_dir': str(config.get_output_dir('03_trend_analysis_monthly_annual')),
    'aoi_pattern': config.PROJECT_SETTINGS['aoi_pattern'],
    'daily_stats_subdir': config.PROJECT_SETTINGS['daily_stats_subdir'],
    'hot_pct': config.PROJECT_SETTINGS['hot_pct']
}

LST_CONFIG = config.LST_CONFIG
TEMPORAL_CONFIG = config.TEMPORAL_CONFIG
VISUALIZATION_CONFIG = config.VISUALIZATION_CONFIG
PROCESSING_OPTIONS = config.PROCESSING_OPTIONS

# Quick validation of LST time selection
valid_lst_times = ['day', 'night', 'both']
if LST_CONFIG['lst_time'] not in valid_lst_times:
    raise ValueError(f"Invalid lst_time in config.py: {LST_CONFIG['lst_time']}. Must be one of {valid_lst_times}")

print(f"✅ Configuration loaded from config.py")
print(f"📅 Time period: {PROJECT_CONFIG['start_date']} to {PROJECT_CONFIG['end_date']}")
print(f"🌡️ LST time mode: {LST_CONFIG['lst_time']}")
print(f"🎯 QA levels to use: {LST_CONFIG['qa_levels_to_use']}")
print(f"📊 Primary collection: {LST_CONFIG['primary_collection']}")
print(f"🔧 Processing options: {PROCESSING_OPTIONS['qa_filtering']} QA filtering")
print("\n⚠️ NOTE: To modify settings, edit config.py and restart the script.")
print("="*70)

<a id="load-aoi"></a>
## Block 3: Load AOI Data

**Purpose**: Auto-load city boundaries from Notebook 01 outputs.

**What This Section Does**:
- Reads AOI GeoJSON files from `data/outputs/01_aoi_delineation/City_*/`
- Extracts bounding boxes for LST extraction
- Validates geometry and CRS compatibility

**Expected Output**: Dictionary of city AOI data ready for GEE queries

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# =============================================================================
# LOAD AOI DATA
# =============================================================================

print("="*70)
print("LOADING AOI BOUNDARIES")
print("="*70)

# Use modularized function to load AOI boundaries
aoi_data = gee_utils.load_aoi_boundaries(
    aoi_results_dir=PROJECT_CONFIG['aoi_results_dir']
)

print(f"\n✅ Loaded AOI data for {len(aoi_data)} cities:")
for city in sorted(aoi_data.keys()):
    bounds = aoi_data[city]['bounds']
    print(f"  📍 {city}: Bounds [{bounds[0]:.3f}, {bounds[1]:.3f}, {bounds[2]:.3f}, {bounds[3]:.3f}]")
    
print("="*70)

<a id="lst-extraction"></a>
## Block 4: LST Extraction with Hierarchical QA Filtering

**Purpose**: Download MODIS LST data for all cities with quality-assured filtering.

**Parallel Processing Strategy:**
- Dask workers process multiple cities simultaneously
- Dynamic resource allocation based on CPU cores
- Real-time progress tracking with `tqdm`

**Hierarchical Quality Assurance (QA):**

MODIS LST products include QA bands with error estimates:

| QA Level | Bits | Error Threshold | Description |
|----------|------|-----------------|-------------|
| **Strict** | 00 | ≤ 1K | Highest quality pixels |
| **Moderate** | 01 | ≤ 2K | Medium quality pixels |
| ~~Loose~~ | ~~10~~ | ~~≤ 3K~~ | Not used in this study |

**Fallback Logic:**
1. Extract LST using Strict QA mask
2. If insufficient data → Apply Moderate QA mask
3. Merge results with Strict taking precedence

**Output Products:**
- Monthly climatology GeoTIFFs (12 months)
- Yearly LST GeoTIFFs (2006–2025)
- QA composition maps (percentage of Strict/Moderate pixels)

**Processing Time:** ~5–15 minutes per city (depends on AOI size and time range)

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 12: load_modis_collection_with_hierarchical_qa
# Function definition moved to src/gee_utils.py
# ─────────────────────────────────────────────────────────────────────────────
from src import gee_utils

def load_modis_collection_with_hierarchical_qa(city_name, geometry, lst_time):
    """Thin wrapper — delegates to gee_utils.load_modis_collection_with_hierarchical_qa."""
    return gee_utils.load_modis_collection_with_hierarchical_qa(
        city_name=city_name,
        geometry=geometry,
        lst_time=lst_time,
        lst_config=LST_CONFIG,
        temporal_config=TEMPORAL_CONFIG,
        project_config=PROJECT_CONFIG,
    )

print("✅ load_modis_collection_with_hierarchical_qa loaded from gee_utils.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 13: compute_and_export_lst_geotiff_with_qa
# Function definition moved to src/gee_utils.py
# ─────────────────────────────────────────────────────────────────────────────

def compute_and_export_lst_geotiff_with_qa(collection, geometry, city_name,
                                           city_dir, period_type, period_value,
                                           lst_time, lst_config=None,
                                           project_config=None):
    """Thin wrapper — delegates to gee_utils.compute_and_export_lst_geotiff_with_qa."""
    return gee_utils.compute_and_export_lst_geotiff_with_qa(
        collection=collection,
        geometry=geometry,
        city_name=city_name,
        city_dir=city_dir,
        period_type=period_type,
        period_value=period_value,
        lst_time=lst_time,
        lst_config=lst_config or LST_CONFIG,
        project_config=project_config or PROJECT_CONFIG,
    )

print("✅ compute_and_export_lst_geotiff_with_qa loaded from gee_utils.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 14: Dask-based parallel GEE downloads
# process_city_period() is a local worker function (requires Dask serialisation).
# init_ee_in_worker() and the QA filtering logic mirror gee_utils.py.
# ─────────────────────────────────────────────────────────────────────────────

import os
import time
import traceback
import requests
from tqdm.notebook import tqdm
import ee
from dask.distributed import Client, as_completed
from dask import delayed


def init_ee_in_worker():
    """Initialize Earth Engine in each Dask worker."""
    import sys
    from pathlib import Path
    try:
        ee.Number(1).getInfo()
    except Exception:
        project_root = (Path(os.getcwd()).parent
                        if Path(os.getcwd()).name == 'notebooks'
                        else Path(os.getcwd()))
        sys.path.insert(0, str(project_root))
        import config
        from google.oauth2 import service_account
        creds = service_account.Credentials.from_service_account_file(
            config.GEE_CONFIG['service_json'],
            scopes=['https://www.googleapis.com/auth/earthengine'])
        ee.Initialize(credentials=creds, project=config.GEE_CONFIG['project_id'])


def process_city_period(city_name, bounds_list, city_dir, period_type,
                        period_value, lst_time,
                        lst_config, temporal_config, project_config):
    """
    Worker function: download one period's LST + QA GeoTIFFs.
    Mirrors gee_utils.compute_and_export_lst_geotiff_with_qa but operates
    on a serialisable bounds_list instead of an ee.Geometry.
    """
    try:
        init_ee_in_worker()
        start_time = time.time()

        city_geometry = ee.Geometry.Rectangle(bounds_list)
        result = {'city': city_name, 'period_type': period_type,
                  'period_value': period_value, 'lst_time': lst_time,
                  'status': 'pending'}

        if period_type == 'yearly':
            file_name    = f"{city_name}_LST_{lst_time.title()}_Yearly_{period_value}"
            qa_file_name = f"{city_name}_QA_Source_{lst_time.title()}_Yearly_{period_value}"
        else:
            file_name    = (f"{city_name}_LST_{lst_time.title()}"
                            f"_MonthlyClimatology_{period_value:02d}")
            qa_file_name = (f"{city_name}_QA_Source_{lst_time.title()}"
                            f"_MonthlyClimatology_{period_value:02d}")

        lst_subfolder   = 'Day' if lst_time == 'day' else 'Night'
        city_geotiff_dir = os.path.join(
            project_config['lst_output_dir'], city_dir,
            'LST_Temporal_Analysis', lst_subfolder, 'geotiff')
        os.makedirs(city_geotiff_dir, exist_ok=True)

        file_path    = os.path.join(city_geotiff_dir, f"{file_name}.tif")
        qa_file_path = os.path.join(city_geotiff_dir, f"{qa_file_name}.tif")

        if os.path.exists(file_path):
            result['status'] = 'skipped'
            result['message'] = f"File already exists: {file_path}"
            return result

        cc = lst_config['collections'][lst_config['primary_collection']]
        col = (ee.ImageCollection(cc['id'])
               .filterDate(project_config['start_date'], project_config['end_date'])
               .filterBounds(city_geometry))

        def apply_qa(image):
            lst = image.select(cc[lst_time + '_band'])
            qa  = image.select(cc[lst_time + '_qa_band'])
            lc  = lst.multiply(cc['scale_factor']).add(cc['offset'])
            qh  = lst_config['quality_hierarchy']
            ql  = lst_config['qa_levels_to_use']
            sorted_ql = sorted(ql, key=lambda t: qh[t]['priority'])
            qa_src = ee.Image(0).rename('QA_Source')
            lh = None
            for i, tier in enumerate(sorted_ql):
                mask = qa.bitwiseAnd(3).eq(qh[tier]['qa_bits'])
                qa_src = qa_src.where(mask, qh[tier]['priority'])
                if i == 0:
                    lh = lc.updateMask(mask)
                else:
                    lh = lh.blend(lc.updateMask(mask))
            if lh is None:
                return ee.Image([]).copyProperties(image, ['system:time_start'])
            lh = lh.rename('LST')
            vm = lh.gt(5).And(lh.lt(70))
            return ee.Image([lh.updateMask(vm),
                             qa_src.updateMask(vm)]).copyProperties(
                                 image, ['system:time_start'])

        processed = col.map(apply_qa)
        if period_type == 'yearly':
            filt = processed.filterDate(f"{period_value}-01-01",
                                        f"{period_value}-12-31")
        else:
            filt = processed.filter(
                ee.Filter.calendarRange(period_value, period_value, 'month'))

        count = filt.size().getInfo()
        if count == 0:
            result['status'] = 'no_data'
            result['message'] = (f"No images for {city_name} "
                                 f"{period_type}={period_value} ({lst_time})")
            return result

        lst_comp = filt.select('LST').median()
        qa_comp  = filt.select('QA_Source').mode()
        params   = {'scale': lst_config['scale'], 'crs': 'EPSG:4326',
                    'region': city_geometry, 'format': 'GEO_TIFF'}
        lst_url  = lst_comp.getDownloadURL(params)
        qa_url   = qa_comp.getDownloadURL(params)

        def _dl(url, path, retries=3):
            for i in range(retries):
                try:
                    r = requests.get(url, timeout=60)
                    if r.status_code == 200:
                        with open(path, 'wb') as f:
                            f.write(r.content)
                        return
                except Exception as exc:
                    if i == retries - 1:
                        raise exc
                    time.sleep(2)

        _dl(lst_url, file_path)
        _dl(qa_url, qa_file_path)

        result.update({'status': 'downloaded', 'file_path': file_path,
                       'qa_file_path': qa_file_path,
                       'duration': time.time() - start_time,
                       'image_count': count})
        return result

    except Exception as exc:
        return {'city': city_name, 'period_type': period_type,
                'period_value': period_value, 'lst_time': lst_time,
                'status': 'error', 'message': str(exc),
                'traceback': traceback.format_exc()}


# ── Main execution ─────────────────────────────────────────────────────────
print("\n🚀 Starting Dask-based LST Data Processing")
print("="*80)

client.run(init_ee_in_worker)

tasks = []
lst_time_options = (['day', 'night'] if LST_CONFIG['lst_time'] == 'both'
                    else [LST_CONFIG['lst_time']])

for city_name, aoi_info in aoi_data.items():
    bounds       = aoi_info['bounds']
    bounds_list  = bounds.tolist() if hasattr(bounds, 'tolist') else list(bounds)
    city_dir     = aoi_info['city_dir']
    for lst_time in lst_time_options:
        for year in TEMPORAL_CONFIG['years']:
            tasks.append({'city_name': city_name, 'bounds_list': bounds_list,
                          'city_dir': city_dir, 'period_type': 'yearly',
                          'period_value': year, 'lst_time': lst_time})
        for month in TEMPORAL_CONFIG['months']:
            tasks.append({'city_name': city_name, 'bounds_list': bounds_list,
                          'city_dir': city_dir,
                          'period_type': 'monthly_climatology',
                          'period_value': month, 'lst_time': lst_time})

print(f"Created {len(tasks)} tasks")
print(f"Cities: {', '.join(aoi_data.keys())}")
print(f"Years:  {TEMPORAL_CONFIG['years'][0]}–{TEMPORAL_CONFIG['years'][-1]}")
print(f"LST:    {', '.join(lst_time_options)}")

n_workers = len(client.scheduler_info()['workers'])
n_cities_this_run = len(aoi_data)
print(f"\n📊 Submitting tasks to Dask cluster  ({n_workers} workers)…")

dask_tasks = [
    delayed(process_city_period)(
        t['city_name'], t['bounds_list'], t['city_dir'],
        t['period_type'], t['period_value'], t['lst_time'],
        dict(LST_CONFIG), dict(TEMPORAL_CONFIG), dict(PROJECT_CONFIG))
    for t in tasks]

futures = client.compute(dask_tasks)
results = []
errors  = []

with tqdm(total=len(tasks), desc="Processing LST data") as pbar:
    for future, result in as_completed(futures, with_results=True):
        results.append(result)
        if result['status'] == 'downloaded':
            pbar.set_postfix_str(
                f"✅ {result['city']} {result['period_type']}={result['period_value']}")
        elif result['status'] == 'skipped':
            pbar.set_postfix_str(f"⏭️  {result['city']}")
        elif result['status'] == 'error':
            errors.append(result)
            pbar.set_postfix_str(f"❌ {result['city']}")
        pbar.update(1)

downloaded = [r for r in results if r['status'] == 'downloaded']
skipped    = [r for r in results if r['status'] == 'skipped']
no_data    = [r for r in results if r['status'] == 'no_data']

print("\n" + "="*80)
print("🎯 PROCESSING SUMMARY")
print(f"  Total    : {len(results)}")
print(f"  Downloaded: {len(downloaded)}")
print(f"  Skipped  : {len(skipped)}")
print(f"  No data  : {len(no_data)}")
print(f"  Errors   : {len(errors)}")

if errors:
    print("\n❌ ERRORS (first 5):")
    for e in errors[:5]:
        print(f"  {e['city']} {e['period_type']}={e['period_value']}: {e['message']}")

print("\n✅ Processing complete!")
print(f"Output dir: {PROJECT_CONFIG['lst_output_dir']}")


<a id="visualize-maps"></a>
## Block 5: Visualize LST Maps

**Purpose**: Generate publication-ready maps in A5 format (5.83" × 8.27").

**Monthly Climatology Maps:**
- 12 subplots (one per month) in a single figure
- Color scale: Perceptually uniform "RdYlBu_r"
- Consistent scale across all months for comparability

**Yearly LST Maps:**
- 20 subplots (2006–2025) in a single figure
- Tracks inter-annual temperature variations
- Identifies anomalously warm/cool years

**QA Composition Maps:**
- Visualizes spatial distribution of QA levels
- Color coding: Green (Strict) → Yellow (Moderate) → Red (Poor/No Data)

**Output Formats:** PNG (screen viewing) and PDF (publication-ready)

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 16: Dask-based monthly + yearly LST map visualisation
# create_monthly_climatology_map / create_yearly_lst_map → src/visualization.py
# ─────────────────────────────────────────────────────────────────────────────
import glob as _glob
from dask import delayed
from dask.distributed import as_completed
from src import visualization
from tqdm.notebook import tqdm

# ── Skip logic ────────────────────────────────────────────────────────────────
def _viz_outputs_exist(aoi_data, lst_output_dir, lst_time_opts):
    for city_name, info in aoi_data.items():
        city_dir = info['city_dir']
        for lt in lst_time_opts:
            subdir  = 'Day' if lt == 'day' else 'Night'
            fig_dir = os.path.join(lst_output_dir, city_dir,
                                   'LST_Temporal_Analysis', subdir, 'Figures')
            if not _glob.glob(os.path.join(fig_dir,
                              f"{city_name}*Monthly_LST_Climatology.png")):
                return False
            if not _glob.glob(os.path.join(fig_dir,
                              f"{city_name}*Yearly_LST.png")):
                return False
    return True

_dask_lst_opts  = (['day', 'night'] if LST_CONFIG['lst_time'] == 'both'
                   else [LST_CONFIG['lst_time']])
_skip_dask_viz  = _viz_outputs_exist(
    aoi_data, PROJECT_CONFIG['lst_output_dir'], _dask_lst_opts)

if _skip_dask_viz:
    print("="*70)
    print("⏭️  Skipping DASK-BASED VISUALIZATION — figures already exist.")
    print("    Delete a city's Figures/ directory to force regeneration.")
    print("="*70)
else:
    print("="*70)
    print("🗺️  DASK-BASED VISUALIZATION WITH QA STATISTICS")
    print("="*70)

    viz_tasks = []
    lst_time_options = _dask_lst_opts

    for city_name, aoi_info in aoi_data.items():
        city_dir  = aoi_info['city_dir']
        for lst_time in lst_time_options:
            vis_config = VISUALIZATION_CONFIG[lst_time]
            viz_tasks.append(
                delayed(visualization.create_monthly_climatology_map)(
                    city_name, city_dir, lst_time,
                    dict(LST_CONFIG), dict(vis_config),
                    dict(TEMPORAL_CONFIG), dict(PROJECT_CONFIG)))
            viz_tasks.append(
                delayed(visualization.create_yearly_lst_map)(
                    city_name, city_dir, lst_time,
                    dict(LST_CONFIG), dict(vis_config),
                    dict(TEMPORAL_CONFIG), dict(PROJECT_CONFIG)))

    print(f"Created {len(viz_tasks)} visualization tasks  "
          f"({', '.join(aoi_data.keys())})")
    print(f"Dashboard: {client.dashboard_link}")

    futures = client.compute(viz_tasks)
    results = []
    with tqdm(total=len(viz_tasks), desc="Creating visualizations") as pbar:
        for future, result in as_completed(futures, with_results=True):
            results.append(result)
            st = result.get('status', '?')
            if st == 'success':
                pbar.set_postfix_str(
                    f"✅ {result['city']} {result['lst_time']} {result['type']}")
            elif st == 'error':
                pbar.set_postfix_str(
                    f"❌ {result.get('city','?')} {result.get('message','')[:60]}")
            pbar.update(1)

    ok  = [r for r in results if r.get('status') == 'success']
    err = [r for r in results if r.get('status') == 'error']
    nd  = [r for r in results if r.get('status') == 'no_data']
    print(f"\n✅ Done — {len(ok)} success, {len(nd)} no data, {len(err)} errors")
    if err:
        for e in err[:5]:
            print(f"  ❌ {e.get('city')} {e.get('lst_time')} {e.get('type')}: "
                  f"{str(e.get('message',''))[:120]}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 17: QA map visualisation
# create_qa_maps() → src/visualization.py
# ─────────────────────────────────────────────────────────────────────────────
import glob as _glob
from src import visualization

# ── Skip logic ────────────────────────────────────────────────────────────────
def _qa_outputs_exist(aoi_data, lst_output_dir, lst_time_opts):
    for city_name, info in aoi_data.items():
        city_dir = info['city_dir']
        for lt in lst_time_opts:
            subdir = 'Day' if lt == 'day' else 'Night'
            qa_dir = os.path.join(lst_output_dir, city_dir,
                                  'LST_Temporal_Analysis', subdir,
                                  'Figures', 'QA_Maps')
            if not _glob.glob(os.path.join(qa_dir,
                              f"{city_name}*Monthly_QA_Climatology.png")):
                return False
            if not _glob.glob(os.path.join(qa_dir,
                              f"{city_name}*Yearly_QA.png")):
                return False
    return True

_qa_lst_opts = (['day', 'night'] if LST_CONFIG['lst_time'] == 'both'
                else [LST_CONFIG['lst_time']])
_skip_qa_viz = _qa_outputs_exist(
    aoi_data, PROJECT_CONFIG['lst_output_dir'], _qa_lst_opts)

if _skip_qa_viz:
    print("="*70)
    print("⏭️  Skipping QA MAP VISUALIZATION — QA figures already exist.")
    print("    Delete a city's Figures/QA_Maps/ directory to force regeneration.")
    print("="*70)
else:
    print("="*70)
    print("🗺️  QA MAP VISUALIZATION")
    print("="*70)
    visualization.create_qa_maps(
        aoi_data=aoi_data,
        lst_config=LST_CONFIG,
        temporal_config=TEMPORAL_CONFIG,
        project_config=PROJECT_CONFIG,
    )
    print("\n✅ QA map visualisation complete.")
    print("="*70)


<a id="compute-stats"></a>
## Block 6: Compute LST Statistics

**Purpose**: Extract comprehensive statistics from GeoTIFF files.

**Data Filtering:**
- Excludes LST < 10°C (likely water bodies, cloud artifacts, no-data)
- Retains only valid land surface temperatures

**Computed Metrics:**

For each city, month/year, and time period (day/night):

| Metric | Formula | Description |
|--------|---------|-------------|
| Mean | $\bar{x} = \frac{1}{n}\sum_{i=1}^{n}x_i$ | Average temperature |
| Std Dev | $\sigma = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2}$ | Temperature variability |
| Min/Max | — | Temperature extremes |
| Range | $R = x_{max} - x_{min}$ | Thermal amplitude |
| Median | $\tilde{x}$ | 50th percentile |

**Output Files:**
- Per-city CSV: `City_*/LST_Temporal_Analysis/[Day/Night]/Statistics/`
- Combined CSV: `Stats/All_Cities_LST_Statistics.csv`

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 6: LST SUMMARY WITH YEARLY STATISTICS
# compute_lst_statistics_from_files / build_enhanced_city_summary
# → src/temporal_analysis.py
# ─────────────────────────────────────────────────────────────────────────────
print("="*70)
print("BLOCK 6: LST SUMMARY WITH YEARLY STATISTICS")
print("="*70)

from src import temporal_analysis

main_stats_dir     = os.path.join(PROJECT_CONFIG['lst_output_dir'], 'Stats')
os.makedirs(main_stats_dir, exist_ok=True)
_stats_skip_file   = os.path.join(main_stats_dir, 'All_Cities_LST_Statistics.csv')
_summary_skip_file = os.path.join(main_stats_dir, 'City_LST_Enhanced_Summary.csv')

_skip_b6 = os.path.exists(_stats_skip_file)
if _skip_b6:
    print("⏭️  Skipping Block 6 — output files already exist.")
    print(f"    Stats  : {_stats_skip_file}")
    print(f"    Summary: {_summary_skip_file}")
    print("    Delete those files to force a full rebuild.")
    all_stats_file   = _stats_skip_file
    all_stats_df     = pd.read_csv(all_stats_file)
    if os.path.exists(_summary_skip_file):
        summary_df       = pd.read_csv(_summary_skip_file)
        summary_csv_path = _summary_skip_file
        print(f"    ✅ Loaded summary ({len(summary_df)} rows) "
              f"and stats ({len(all_stats_df)} rows)")
    final_stats_duration = 0.0
    print("="*70)

if not _skip_b6:
    import time as _time_b6
    _t0 = _time_b6.time()

    lst_times = (['day', 'night'] if LST_CONFIG['lst_time'] == 'both'
                 else [LST_CONFIG['lst_time']])

    # ── Read all GeoTIFFs and build per-pixel statistics ──────────────────
    records = temporal_analysis.compute_lst_statistics_from_files(
        aoi_data=aoi_data,
        lst_output_dir=PROJECT_CONFIG['lst_output_dir'],
        temporal_config=TEMPORAL_CONFIG,
        lst_config=LST_CONFIG,
    )

    if not records:
        print("⚠️  No GeoTIFF data found — check the output directory.")
    else:
        all_stats_df   = pd.DataFrame(records)
        all_stats_file = _stats_skip_file
        all_stats_df.to_csv(all_stats_file, index=False)
        print(f"✅ Saved All_Cities_LST_Statistics.csv  ({len(all_stats_df)} rows)")

        # ── Build enhanced summary with trends ───────────────────────────
        summary_df = temporal_analysis.build_enhanced_city_summary(
            all_stats_df=all_stats_df,
            temporal_config=TEMPORAL_CONFIG,
            lst_times=lst_times,
        )

        # Add Trend Category column
        summary_df['Trend Category'] = summary_df.apply(
            lambda r: temporal_analysis.categorise_trend(
                r.get('Warming Rate', ''),
                r.get('Trend Significance', '')),
            axis=1)

        summary_csv_path = _summary_skip_file
        summary_df.to_csv(summary_csv_path, index=False)
        print(f"✅ Saved City_LST_Enhanced_Summary.csv  ({len(summary_df)} rows)")

    final_stats_duration = _time_b6.time() - _t0
    print(f"⏱️  Block 6 completed in {final_stats_duration:.1f} s")
    print("="*70)


<a id="trend-analysis"></a>
## Block 7: Trend Analysis & Visualization

**Purpose**: Perform statistical trend tests and create diagnostic plots for each city.

Creates dual-panel diagnostic plots for each city:

**Left Panel: Annual Trend**
- Time series of yearly mean LST (2006–2025)
- Linear regression trend line: $y = \beta_0 + \beta_1 \cdot t$
- 95% confidence interval (shaded region)
- Warming rate: $\Delta T = \beta_1 \times 10$ °C/decade
- Statistical significance: $p$-value and $R^2$
- Mann-Kendall test: $Z$-statistic and trend direction

**Right Panel: Monthly Climatology**
- Seasonal cycle (12 months)
- Shaded envelope: Min–Max temperature range
- Annotated peaks: Hottest and coolest months
- Summary statistics in text box

**Color Scheme:**
- Day LST: Orange (#FF7F0E)
- Night LST: Black (#000000)
- Confidence bands: Semi-transparent fill

**Significance Markers:**
- (*) indicates $p < 0.05$
- Mann-Kendall trend: ↑ (increasing), ↓ (decreasing), → (no trend)

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 7: LST TREND ANALYSIS — IMPROVED VISUALIZATION
# create_lst_trend_plots() → src/visualization.py
# ─────────────────────────────────────────────────────────────────────────────
import glob as _glob
from src import visualization

# ── SKIP LOGIC ────────────────────────────────────────────────────────────────
# Block 7 creates per-city trend figures (*_LST_Trends_Combined2.png) inside
# {lst_output_dir}/City_{name}/LST_Temporal_Analysis/Figures/ and also relies
# on City_LST_Enhanced_Summary.csv (produced by Block 6).
# Skip if the summary CSV exists AND the first city already has its trend PNG.
# Delete the summary CSV or a city's Trend PNGs to force a rebuild.
# ──────────────────────────────────────────────────────────────────────────────

_trend_summary_file = os.path.join(
    PROJECT_CONFIG['lst_output_dir'], 'Stats', 'City_LST_Enhanced_Summary.csv')
_skip_b7 = False
if os.path.exists(_trend_summary_file):
    _first_city = list(aoi_data.keys())[0]
    # Look for trend figures under the *LST output* tree, NOT the AOI tree
    _trend_figs = _glob.glob(os.path.join(
        PROJECT_CONFIG['lst_output_dir'], f'City_{_first_city}',
        'LST_Temporal_Analysis', '**', 'Figures',
        f'{_first_city}_LST_Trends_Combined2.png'), recursive=True)
    if _trend_figs:
        _skip_b7 = True

if _skip_b7:
    print("="*70)
    print("⏭️  Skipping Block 7 — trend figures & summary already exist.")
    print(f"    Summary: {_trend_summary_file}")
    print("    Delete the summary CSV or a city's Trend PNGs to force rebuild.")
    summary_df       = pd.read_csv(_trend_summary_file)
    summary_csv_path = _trend_summary_file
    print(f"    ✅ Loaded summary_df ({len(summary_df)} rows)")
    print("="*70)

if not _skip_b7:
    print("="*70)
    print("BLOCK 7: LST TREND ANALYSIS — IMPROVED VISUALIZATION")
    print("="*70)

    # Load stats file
    _stats_file = os.path.join(
        PROJECT_CONFIG['lst_output_dir'], 'Stats', 'All_Cities_LST_Statistics.csv')
    if os.path.exists(_stats_file):
        all_stats_df = pd.read_csv(_stats_file)
        print(f"✅ Loaded stats from {_stats_file} ({len(all_stats_df)} rows)")
    else:
        raise FileNotFoundError(
            f"All_Cities_LST_Statistics.csv not found. Run Block 6 first.\n"
            f"Expected: {_stats_file}")

    visualization.create_lst_trend_plots(
        aoi_data=aoi_data,
        all_stats_df=all_stats_df,
        temporal_config=TEMPORAL_CONFIG,
        project_config=PROJECT_CONFIG,
        lst_config=LST_CONFIG,
    )

    print("\n✅ Block 7 complete — trend plots saved for all cities.")
    print("="*70)

<a id="heatmap"></a>
## Block 8: Cross-City Heatmap Comparison

**Purpose**: Create comparative heatmaps revealing spatiotemporal patterns across all cities.

**Yearly Heatmap (Cities × Years):**
- Rows: Cities (sorted warmest to coolest)
- Columns: Years (2006–2025)
- Identifies warming/cooling trends across urban areas
- Highlights anomalous years (e.g., El Niño events)

**Monthly Heatmap (Cities × Months):**
- Rows: Cities
- Columns: Months (Jan–Dec)
- Reveals latitudinal climate gradients
- Compares peak temperature timing across cities

**Color Scale:**
- Diverging palette (RdYlBu_r) centred on median
- Consistent scaling for fair comparison
- Separate heatmaps for day and night LST

**Pagination:** When >5 cities, heatmaps are paginated (10 cities per page) to maintain readability.

**Output:** Saved to `Stats/` folder

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# ── SKIP LOGIC ────────────────────────────────────────────────────────────────
# Heatmap cell creates LST_Yearly_*_Heatmap_AllCities*.png files in Stats/.
# Skip if at least one such PNG already exists.
# Delete the heatmap PNGs from Stats/ to force a full rebuild.
import glob as _glob

_stats_dir = os.path.join(
    config.get_output_dir('03_trend_analysis_monthly_annual'), 'Stats')
_existing_heatmaps = _glob.glob(
    os.path.join(_stats_dir, 'LST_Yearly_*_Heatmap_AllCities*.png'))
_skip_heatmap = len(_existing_heatmaps) > 0

if _skip_heatmap:
    print("="*70)
    print("⏭️  Skipping Heatmap — paginated heatmap PNGs already exist.")
    print(f"    Found {len(_existing_heatmaps)} existing heatmap file(s) in:")
    print(f"    {_stats_dir}")
    print("    Delete those PNGs to force a rebuild.")
    cross_city_stats_dir = _stats_dir
    print("="*70)

if not _skip_heatmap:
    # =============================================================================
    # HEATMAP VISUALIZATION: LST ACROSS CITIES AND TIME
    # =============================================================================

    print("="*70)
    print("CREATING LST HEATMAPS ACROSS CITIES")
    print("="*70)

    # Determine which LST times to process
    lst_times_to_process = []
    if config.LST_CONFIG['lst_time'] in ['day', 'both']:
        lst_times_to_process.append('day')
    if config.LST_CONFIG['lst_time'] in ['night', 'both']:
        lst_times_to_process.append('night')

    # Load the combined statistics file
    combined_stats_file = os.path.join(
        config.get_output_dir('03_trend_analysis_monthly_annual'),
        'Stats',
        'All_Cities_LST_Statistics.csv'
    )

    if not os.path.exists(combined_stats_file):
        raise FileNotFoundError(f"Combined statistics file not found: {combined_stats_file}")

    all_stats_df = pd.read_csv(combined_stats_file)

    # Prepare data for heatmaps
    yearly_heatmaps = {}
    monthly_heatmaps = {}

    for lst_time in lst_times_to_process:
        # Filter data for this LST time
        lst_data = all_stats_df[all_stats_df['day_night'] == lst_time].copy()

        # Collect yearly and monthly data by city
        city_yearly_data = {}
        city_monthly_data = {}

        for city_name in aoi_data.keys():
            # Filter for this city
            city_data = lst_data[lst_data['city'] == city_name].copy()

            if not city_data.empty:
                # Yearly data: non-climatology yearly records
                yearly_df = city_data[
                    (city_data['time_period_type'] == 'yearly') &
                    (city_data['is_climatology'] == False)
                ].copy()

                # Monthly data: USE CLIMATOLOGY (monthly averages across all years)
                monthly_df = city_data[
                    (city_data['time_period_type'] == 'monthly') &
                    (city_data['is_climatology'] == True)  # Changed to True for climatology
                ].copy()

                # Rename and convert columns to match expected format for create_heatmap_data
                if not yearly_df.empty:
                    yearly_df = yearly_df.rename(columns={'time_period': 'year', 'lst_median': 'median'})
                    yearly_df['year'] = yearly_df['year'].astype(int)  # Ensure year is integer
                    city_yearly_data[city_name] = yearly_df

                if not monthly_df.empty:
                    monthly_df = monthly_df.rename(columns={'time_period': 'month', 'lst_median': 'median'})
                    monthly_df['month'] = monthly_df['month'].astype(int)  # Ensure month is integer
                    city_monthly_data[city_name] = monthly_df
            else:
                print(f"⚠️ No statistics data found for {city_name} ({lst_time})")

        # Create heatmap data
        if city_yearly_data:
            yearly_heatmaps[lst_time] = temporal_analysis.create_heatmap_data(
                city_yearly_data,
                time_dim='year',
                stat='median'
            )
        else:
            yearly_heatmaps[lst_time] = pd.DataFrame()

        if city_monthly_data:
            monthly_heatmaps[lst_time] = temporal_analysis.create_heatmap_data(
                city_monthly_data,
                time_dim='month',
                stat='median'
            )
        else:
            monthly_heatmaps[lst_time] = pd.DataFrame()

        print(f"✅ Prepared {lst_time}-time heatmap data")

    # Create output directory for cross-city statistics
    cross_city_stats_dir = os.path.join(
        config.get_output_dir('03_trend_analysis_monthly_annual'),
        'Stats'
    )

    # =============================================================================
    # HEATMAP SETTINGS
    # =============================================================================
    # - High DPI (400) for publication quality
    # - Cities sorted warmest to coolest (warmest at TOP)
    # - If ≤5 cities: single compact heatmap (original format)
    # - If >5 cities: paginated with 10 cities per page, sorted order preserved
    #   Files named: LST_Yearly_Day_Heatmap_AllCities_Page1of5.png, etc.
    # =============================================================================
    HEATMAP_DPI = 400
    CITIES_PER_PAGE = 10  # Adjust to 20 after reviewing first output if needed

    # Create and save SEPARATE heatmaps for each LST time and period
    for lst_time in lst_times_to_process:
        # Yearly heatmap (paginated if >5 cities)
        if not yearly_heatmaps[lst_time].empty:
            yearly_output = os.path.join(
                cross_city_stats_dir,
                f'LST_Yearly_{lst_time.title()}_Heatmap_AllCities.png'
            )
            visualization.create_lst_heatmap(
                yearly_heatmaps[lst_time],
                lst_time=lst_time,
                time_dim='year',
                output_path=yearly_output,
                dpi=HEATMAP_DPI,
                cities_per_page=CITIES_PER_PAGE
            )
        else:
            print(f"⚠️ Skipping yearly {lst_time} heatmap - no data available")

        # Monthly climatology heatmap (paginated if >5 cities)
        if not monthly_heatmaps[lst_time].empty:
            monthly_output = os.path.join(
                cross_city_stats_dir,
                f'LST_Monthly_{lst_time.title()}_Heatmap_AllCities.png'
            )
            visualization.create_lst_heatmap(
                monthly_heatmaps[lst_time],
                lst_time=lst_time,
                time_dim='month',
                output_path=monthly_output,
                dpi=HEATMAP_DPI,
                cities_per_page=CITIES_PER_PAGE
            )
        else:
            print(f"⚠️ Skipping monthly {lst_time} heatmap - no data available")

    print("\n" + "="*70)
    print("✅ LST HEATMAP ANALYSIS COMPLETED")
    print(f"📊 Heatmaps saved to: {cross_city_stats_dir}")
    print("="*70)


<a id="trend-charts"></a>
## Block 9a: Trend Category Summary Chart

**Purpose**: Visualize how many cities fall into each trend category.

Generates a grouped bar chart showing city distribution across:
- Significant Warming
- Significant Cooling
- Insignificant Warming
- No Trend

Displayed for both Day and Night LST trends.

**Output:** `Stats/Trend_Category_Summary_Chart.png` and `.pdf`

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# =============================================================================
# BLOCK 9a: TREND CATEGORY SUMMARY CHART
# create_trend_category_chart() → src/visualization.py
# =============================================================================

_stats_dir_9a    = os.path.join(
    config.get_output_dir('03_trend_analysis_monthly_annual'), 'Stats')
_trend_chart_png = os.path.join(_stats_dir_9a, 'Trend_Category_Summary_Chart.png')
_skip_9a         = os.path.exists(_trend_chart_png)

if _skip_9a:
    print("="*70)
    print("⏭️  Skipping Block 9a — Trend Category Summary Chart already exists.")
    print(f"    {_trend_chart_png}")
    print("    Delete the PNG to force a rebuild.")
    print("="*70)

if not _skip_9a:
    from src import visualization as _viz9a
    _summary_file = os.path.join(_stats_dir_9a, 'City_LST_Enhanced_Summary.csv')
    if not os.path.exists(_summary_file):
        print(f"❌ Summary file not found: {_summary_file}")
        print("   Run Block 7 first to generate City_LST_Enhanced_Summary.csv")
    else:
        print("="*70)
        print("BLOCK 9a: TREND CATEGORY SUMMARY CHART")
        print("="*70)
        _viz9a.create_trend_category_chart(
            summary_csv_path=_summary_file,
            output_dir=_stats_dir_9a,
        )
        print("="*70)
        print("🎯 BLOCK 9a COMPLETED")
        print("="*70)


## Block 9b: Annual LST Violin Chart

**Purpose**: Show the full distribution of annual median LST for each city.

Generates a butterfly (back-to-back) violin chart:
- Cities sorted warmest → right
- Red dot marks the hottest year; blue dot marks the coolest year
- Day and night violins shown side by side

**Output:** `Stats/Annual_LST_Violin_Chart.png` and `.pdf`

In [ ]:
# =============================================================================
# BLOCK 9b: ANNUAL LST VIOLIN CHART  (butterfly / back-to-back)
# create_annual_lst_violin_chart() → src/visualization.py
# =============================================================================

_stats_dir_9b  = os.path.join(
    config.get_output_dir('03_trend_analysis_monthly_annual'), 'Stats')
_violin_png    = os.path.join(_stats_dir_9b, 'Annual_LST_Violin_Chart.png')
_skip_9b       = os.path.exists(_violin_png)

if _skip_9b:
    print("="*70)
    print("⏭️  Skipping Block 9b — Annual LST Violin Chart already exists.")
    print(f"    {_violin_png}")
    print("    Delete the PNG to force a rebuild.")
    print("="*70)

if not _skip_9b:
    from src import visualization as _viz9b
    _all_stats_file = os.path.join(_stats_dir_9b, 'All_Cities_LST_Statistics.csv')
    if not os.path.exists(_all_stats_file):
        print(f"❌ Statistics file not found: {_all_stats_file}")
        print("   Run Block 6/7 first to generate All_Cities_LST_Statistics.csv")
    else:
        print("="*70)
        print("BLOCK 9b: ANNUAL LST VIOLIN CHART  (butterfly layout)")
        print("="*70)
        _viz9b.create_annual_lst_violin_chart(
            all_stats_csv_path=_all_stats_file,
            output_dir=_stats_dir_9b,
            lst_config=LST_CONFIG,
        )
        print("="*70)
        print("🎯 BLOCK 9b COMPLETED")
        print("="*70)


<a id="validation-yearly"></a>
## Block 9: Stakeholder Validation — Yearly Trends

**Purpose**: Validate measured LST trends against stakeholder perceptions of warming.

**Survey Question:**  
*"Have you perceived temperature increases in the last 10 years?"*

**Validation Metrics:**

$$\text{Agreement (\%)} = \frac{\text{Stakeholders perceiving warming}}{\text{Total stakeholders}} \times 100$$

**Classification:**
- **Agree**: Perception matches measured trend direction
- **Disagree**: Perception contradicts measured trend
- **Analysis**: Separate validation for day and night LST

**Output Products:**
- Stakeholder validation sunburst chart
- Per-city agreement tables
- Summary statistics on perception accuracy

**Interpretation:**  
High agreement indicates stakeholders accurately perceive thermal changes. Low agreement may reflect adaptation to gradual warming, focus on extreme events vs. mean trends, or differences between perceived and measured temperature.

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# ============================================================================
# STAKEHOLDER VALIDATION: COMPARE PERCEIVED VS. ACTUAL LST TRENDS (YEARLY)
# create_stakeholder_validation_chart() → src/visualization.py
# ============================================================================
import numpy as np
import math
from src import visualization as _viz_sv

print("\n" + "="*70)
print("ANALYZING STAKEHOLDER VALIDATION (YEARLY TRENDS)")
print("="*70)

# ── Load survey data ─────────────────────────────────────────────────────────
survey_file = os.path.join(config.DATA_DIR, 'inputs', 'stakeholder_survey',
                           'master_composite.csv')
print(f"📂 Loading survey data from: {survey_file}")
survey_df = pd.read_csv(survey_file)
print(f"✅ Loaded {len(survey_df)} survey responses")

study_cities = list(aoi_data.keys())
print(f"📍 Study cities: {study_cities}")

# Filter: workshop attendees only
workshop_attendees = survey_df[survey_df['survey status'] == 'Yes'].copy()

# ── Normalize city names to match AOI / config conventions ───────────────────
CITY_NAME_MAP = {"Ado Ekiti": "Ado-Ekiti", "Osogbo": "Oshogbo"}
workshop_attendees['matched_city_y'] = (
    workshop_attendees['matched_city_y'].replace(CITY_NAME_MAP))
workshop_attendees['matched_city_x'] = (
    workshop_attendees['matched_city_x'].replace(CITY_NAME_MAP))

# Fall back to matched_city_x when matched_city_y is null
workshop_attendees['matched_city_y'] = workshop_attendees['matched_city_y'].fillna(
    workshop_attendees['matched_city_x'])

# Deduplicate: one row per PID, first city (consistent with NB02 approach)
workshop_study = (workshop_attendees
                  .groupby('pid')
                  .first()
                  .reset_index())

total_all_workshop = len(workshop_study)           # 115 – all workshop attendees
workshop_study = workshop_study[
    workshop_study['matched_city_y'].isin(study_cities)].copy()

print(f"✅ Total workshop attendees: {total_all_workshop}")
print(f"✅ Matched to study cities:  {workshop_study['pid'].nunique()}")

# ── Build validation_df ──────────────────────────────────────────────────────
summary_file = os.path.join(
    config.get_output_dir('03_trend_analysis_monthly_annual'),
    'Stats', 'City_LST_Enhanced_Summary.csv')
summary_df = pd.read_csv(summary_file)

cross_city_stats_dir = os.path.join(
    config.get_output_dir('03_trend_analysis_monthly_annual'), 'Stats')
os.makedirs(cross_city_stats_dir, exist_ok=True)

total_workshop_attendees = total_all_workshop      # Use all 115 attendees
validation_results = []

for city_name in study_cities:
    city_responses = workshop_study[workshop_study['matched_city_y'] == city_name]
    n_stakeholders = city_responses['pid'].nunique()
    if n_stakeholders > 0:
        city_unique         = city_responses.groupby('pid').first().reset_index()
        pct_of_total        = n_stakeholders / total_workshop_attendees * 100
        n_perceived_warming = (city_unique['61_temp_increase_10yrs'] == 'Yes').sum()
        pct_perceived_warming = n_perceived_warming / n_stakeholders * 100
        duration_counts     = city_unique['56_recency_with_city'].value_counts().to_dict()

        city_sum_day   = summary_df[(summary_df['City'] == city_name) &
                                    (summary_df['Time'] == 'Day')]
        city_sum_night = summary_df[(summary_df['City'] == city_name) &
                                    (summary_df['Time'] == 'Night')]

        def _sig(row_df):
            if row_df.empty:
                return False, 'No data'
            v   = str(row_df['Trend Significance'].values[0]).lower()
            sig = 'significant' in v and 'not significant' not in v
            return sig, 'Significant increase' if sig else 'No significant trend'

        is_sig_day,   day_sig_text   = _sig(city_sum_day)
        is_sig_night, night_sig_text = _sig(city_sum_night)

        if not city_sum_day.empty:
            overall_agreement = (pct_perceived_warming if is_sig_day
                                 else (0 if pct_perceived_warming > 0 else 100))
        else:
            overall_agreement = -1
    else:
        pct_of_total = pct_perceived_warming = overall_agreement = 0
        duration_counts = {}
        day_sig_text = night_sig_text = 'No data'

    validation_results.append({
        'city': city_name,
        'n_stakeholders': n_stakeholders,
        'pct_of_total': pct_of_total,
        'pct_perceived_warming': pct_perceived_warming,
        'overall_agreement': overall_agreement,
        'duration_counts': duration_counts,
        'day_significance': day_sig_text,
        'night_significance': night_sig_text,
    })

validation_df = pd.DataFrame(validation_results)
actual_total_stakeholders = total_all_workshop     # All 115 attendees
validation_df['pct_of_total'] = (
    validation_df['n_stakeholders'] / actual_total_stakeholders * 100)

print("\n" + "="*70)
print(f"STAKEHOLDER VALIDATION SUMMARY (Total attendees: {actual_total_stakeholders}):")
print("="*70)
print(validation_df[['city', 'n_stakeholders', 'pct_of_total',
                      'pct_perceived_warming', 'overall_agreement']]
      .to_string(index=False))

# ── Create chart ──────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("CREATING STAKEHOLDER VALIDATION SUNBURST CHART…")
print("="*70)

_viz_sv.create_stakeholder_validation_chart(
    validation_df=validation_df,
    output_dir=cross_city_stats_dir,
    chart_type='annual',
)

validation_chart_path = os.path.join(cross_city_stats_dir,
                                     'Stakeholder_Validation_Sunburst_Chart.png')
print(f"\n✅ Stakeholder validation sunburst chart saved:\n   {validation_chart_path}")
print("\n" + "="*70)
print("✅ STAKEHOLDER VALIDATION COMPLETED")
print("="*70)

<a id="validation-monthly"></a>
## Block 10: Stakeholder Validation — Monthly Peak Timing

**Purpose**: Validate stakeholder perceptions of *when* peak heat occurs against measured hottest months.

**Survey Question:**  
*"Which months do you perceive as the hottest?"*

**Validation Approach:**
- Compare stakeholder-selected peak period against MODIS-measured hottest month
- Assess agreement separately for day and night LST
- Calculate percentage of respondents whose perceived peak overlaps with measured peak

**Output Products:**
- Monthly validation sunburst chart
- Per-city agreement summary with day/night breakdown

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# ============================================================================
# BLOCK 10: MONTHLY TREND STAKEHOLDER VALIDATION
# create_stakeholder_validation_chart(..., chart_type='monthly')
# → src/visualization.py
# ============================================================================
print("\n" + "="*70)
print("MONTHLY TREND STAKEHOLDER VALIDATION")
print("="*70)

city_summary = pd.read_csv(summary_csv_path)
monthly_validation_results = []

MONTH_MAP = {'Jan': 'January', 'Feb': 'February', 'Mar': 'March',
             'Apr': 'April',   'May': 'May',       'Jun': 'June',
             'Jul': 'July',    'Aug': 'August',    'Sep': 'September',
             'Oct': 'October', 'Nov': 'November',  'Dec': 'December'}

MONTH_RANGES = {
    'March – May':          ['March', 'April', 'May'],
    'December – February':  ['December', 'January', 'February'],
    'June – August':        ['June', 'July', 'August'],
    'September – November': ['September', 'October', 'November'],
}

for city in study_cities:
    city_responses = workshop_study[workshop_study['matched_city_y'] == city].copy()
    if len(city_responses) == 0:
        continue
    n_stakeholders = city_responses['pid'].nunique()
    city_unique    = city_responses.groupby('pid').first().reset_index()

    city_lst_day   = city_summary[(city_summary['City'] == city) &
                                   (city_summary['Time'] == 'Day')]
    city_lst_night = city_summary[(city_summary['City'] == city) &
                                   (city_summary['Time'] == 'Night')]
    if city_lst_day.empty and city_lst_night.empty:
        continue

    def _hottest(row_df):
        if row_df.empty:
            return None
        abbr = str(row_df['Hottest Month'].values[0]).split(' ')[0]
        return MONTH_MAP.get(abbr, abbr)

    lst_day_hottest   = _hottest(city_lst_day)
    lst_night_hottest = _hottest(city_lst_night)

    peak_responses = city_unique['59_temp_peak_descr'].dropna()
    agree_count = disagree_count = 0
    agree_day_count = agree_night_count = agree_both_count = 0
    for resp in peak_responses:
        if resp in MONTH_RANGES:
            da = lst_day_hottest   in MONTH_RANGES[resp] if lst_day_hottest   else False
            na = lst_night_hottest in MONTH_RANGES[resp] if lst_night_hottest else False
            if da or na:
                agree_count += 1
                if da and na:  agree_both_count  += 1
                elif da:       agree_day_count   += 1
                else:          agree_night_count += 1
            else:
                disagree_count += 1

    total_r = agree_count + disagree_count
    overall_agreement = (agree_count / total_r * 100) if total_r > 0 else np.nan
    duration_counts   = city_unique['56_recency_with_city'].value_counts().to_dict()

    monthly_validation_results.append({
        'city': city,
        'n_stakeholders': n_stakeholders,
        'lst_day_hottest':   lst_day_hottest   or 'N/A',
        'lst_night_hottest': lst_night_hottest or 'N/A',
        'overall_agreement': overall_agreement,
        'agree_day_only':  agree_day_count,
        'agree_night_only': agree_night_count,
        'agree_both':       agree_both_count,
        'duration_counts':  duration_counts,
    })

monthly_validation_df = pd.DataFrame(monthly_validation_results)
actual_total_monthly  = total_all_workshop           # All 115 attendees
monthly_validation_df['pct_of_total'] = (
    monthly_validation_df['n_stakeholders'] / actual_total_monthly * 100)

print("\n" + "="*70)
print(f"MONTHLY PEAK VALIDATION SUMMARY (Total attendees: {actual_total_monthly})")
print("="*70)
for _, row in monthly_validation_df.iterrows():
    print(f"{row['city']}: n={row['n_stakeholders']} ({row['pct_of_total']:.1f}%)  "
          f"Day={row['lst_day_hottest']}  Night={row['lst_night_hottest']}  "
          f"Agreement={row['overall_agreement']:.1f}%")

# ── Chart ─────────────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("CREATING MONTHLY TREND VALIDATION SUNBURST CHART…")
print("="*70)

from src import visualization as _viz_mv
_out_dir = os.path.join(
    config.get_output_dir('03_trend_analysis_monthly_annual'), 'Stats')
_viz_mv.create_stakeholder_validation_chart(
    validation_df=monthly_validation_df,
    output_dir=_out_dir,
    chart_type='monthly',
)

monthly_output = os.path.join(
    _out_dir, 'Stakeholder_Validation_Monthly_Sunburst_Chart.png')
print(f"✅ Monthly validation sunburst chart saved: {monthly_output}")

# Save monthly validation results to CSV ──────────────────────────────────────
monthly_summary_path = os.path.join(_out_dir,
                                    'Stakeholder_Validation_Monthly_Summary.csv')
monthly_validation_df[[
    'city', 'n_stakeholders', 'pct_of_total',
    'lst_day_hottest', 'lst_night_hottest',
    'overall_agreement', 'agree_day_only', 'agree_night_only', 'agree_both',
]].to_csv(monthly_summary_path, index=False)
print(f"✅ Monthly validation summary saved: {monthly_summary_path}")

print("\n" + "="*70)
print("MONTHLY VALIDATION COMPLETE")
print("="*70)

<a id="summary"></a>
## Summary & Next Steps

[↑ Back to Table of Contents](#Table-of-Contents)

---

### Key Accomplishments

This notebook completed **monthly and annual LST temporal analysis** across all study cities:

#### ✅ Data Extraction
- Extracted MODIS Terra/Aqua LST data for all study cities (2006–2025)
- Applied hierarchical QA filtering (Strict → Moderate fallback)
- Generated monthly climatology and yearly GeoTIFFs

#### ✅ Statistical Analysis
- Computed comprehensive temporal statistics (mean, std, min, max, median)
- Performed linear regression with warming rate (°C/decade)
- Applied Mann-Kendall non-parametric trend tests

#### ✅ Visualization
- Created publication-ready trend and climatology plots
- Generated cross-city heatmaps (yearly and monthly)
- Produced trend category summary and violin charts

#### ✅ Stakeholder Validation
- Validated measured annual trends against perceived warming
- Compared measured peak months against perceived hottest periods
- Created sunburst charts summarizing agreement levels

---

### Key Outputs
- Monthly climatology GeoTIFFs (12 per city)
- Yearly LST GeoTIFFs (20 per city)
- Statistical summaries (CSV files)
- Trend analysis plots (PNG/PDF)
- Cross-city comparison heatmaps
- Stakeholder validation reports

---

### Next Steps: Daily LST Analysis (Notebook 04)

Now that we've established **long-term trends** and **seasonal patterns**, the next analysis focuses on **daily-scale dynamics**:

#### Notebook 04: Daily LST Temporal Analysis

**What It Does**:
- Extracts daily MODIS, ERA5-Land, and MSG SEVIRI LST data
- Detects hot events using 90th percentile thresholds
- Identifies persistent heat episodes (≥2 consecutive days)
- Validates inter-dataset consistency (correlation, bias, RMSE)

**Why This Matters**:
- Monthly/annual averages mask important daily variability
- Hot event detection identifies acute health risk periods
- Multi-dataset comparison builds confidence in findings
- Persistent events have greater public health impact than isolated hot days

---

**📝 Note:** All outputs from this notebook are stored in `data/outputs/03_trend_analysis_monthly_annual/`

---